# CNN Spectrogram Classifier Training — ISTerre Event Classification

**Goal**: train a CNN to classify seismic event type (earthquake / rockslide / ice quake)
from 3-component (Z, N, E) spectrogram images built by `07a_spectrogram_dataset_build.py`
on the ISTerre cluster (that script does the SDS access + instrument response removal +
spectrogram computation — this notebook only trains, since the cluster has no GPU).

**Drive folder expected** — the packed output of `07a_consolidate_for_colab.py`
(run on the cluster after 07a; packs the many small per-sample .npz files into a
few large archives, since Google Drive's FUSE mount handles directories with
tens of thousands of files very poorly):
```
MyDrive/colab_cnn_training_spectrogram/
    spectrograms_train.npz   <- packed images + labels for the train split
    spectrograms_val.npz
    spectrograms_test.npz
    image_list.csv           <- manifest, kept for reference/provenance
    freq_axis.npy             <- shared frequency axis [Hz]
    time_axis.npy             <- shared time axis [s]
```

**Runtime**: Runtime > Change runtime type > T4 GPU

## Cell 1 — Check GPU

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print(result.stdout)
else:
    print('No GPU — go to Runtime > Change runtime type > T4 GPU')

## Cell 2 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

## Cell 3 — Configure paths & hyperparameters
**Edit this cell** to match your Drive folder.

In [ ]:
import os

# -- Edit here -----------------------------------------------------------
DRIVE_BASE      = '/content/drive/MyDrive/colab_cnn_training_spectrogram'
SPEC_TRAIN_PATH = os.path.join(DRIVE_BASE, 'spectrograms_train.npz')
SPEC_VAL_PATH   = os.path.join(DRIVE_BASE, 'spectrograms_val.npz')
SPEC_TEST_PATH  = os.path.join(DRIVE_BASE, 'spectrograms_test.npz')
MANIFEST_CSV    = os.path.join(DRIVE_BASE, 'image_list.csv')
FREQ_AXIS_PATH  = os.path.join(DRIVE_BASE, 'freq_axis.npy')
TIME_AXIS_PATH  = os.path.join(DRIVE_BASE, 'time_axis.npy')
LOG_DIR         = os.path.join(DRIVE_BASE, 'log')

# Must match TARGET_CLASSES in 07a_spectrogram_dataset_build.py
CLASS_NAMES = ['earthquake', 'rockslide', 'ice quake']

EPOCHS        = 60
# Raised from 32: ice quake is only ~3.4% of the training set, so at
# batch_size=32 (~1.1 ice-quake samples/batch on average) many batches had
# 0 ice-quake samples and some had 2+, and with class_weight='balanced'
# giving ice quake ~24x the weight of earthquake, that made the per-batch
# loss/gradient wildly inconsistent -- likely why training was collapsing to
# predicting a single constant class each epoch. A bigger batch smooths this
# out (~4.4 ice-quake samples/batch on average at 128).
BATCH_SIZE    = 128
# Lowered from 1e-3: at 1e-3, combined with class_weight='balanced' heavily
# up-weighting the rare ice-quake class (~24x vs earthquake), training was
# collapsing to predicting a single constant class per epoch (val_accuracy
# landing exactly on one class's population fraction, e.g. 0.0357 = ice-quake
# share of val, 0.1316 = rockslide share) instead of learning real features.
LEARNING_RATE = 3e-4
DROPOUT_RATE  = 0.5

# 'custom'   -> the small from-scratch CNN in Cell 9 (build_cnn)
# 'resnet50' -> ImageNet-pretrained ResNet50 backbone, fine-tuned on our
#               spectrograms (build_resnet50_transfer, Cell 9). See the Cell 9
#               markdown for how this relates to Dong et al. (2023)'s method.
MODEL_BACKBONE = 'custom'

# Only used when MODEL_BACKBONE == 'resnet50'. Lower than LEARNING_RATE above:
# fine-tuning a pretrained backbone with a normal from-scratch LR risks wrecking
# the pretrained weights in the first few steps (large gradients through a
# freshly-initialized head backprop into still-adapting pretrained filters).
FINE_TUNE_LEARNING_RATE = 1e-4

# How many of ResNet50's final layers (out of 175 total) are left trainable.
# Everything before that is frozen. This mirrors Dong et al.'s strategy of
# freezing the generic low/mid-level filters (edges, textures -- these transfer
# reasonably well even from natural images to spectrograms) and only
# fine-tuning the last conv block + a new classification head.
RESNET_UNFREEZE_LAST_N = 15
# --------------------------------------------------------------------------

os.makedirs(LOG_DIR, exist_ok=True)

# NOTE: only single-file existence/size checks here — no os.listdir() on a
# directory. Drive's FUSE mount raises "OSError: [Errno 5] Input/output
# error" when listing folders with tens of thousands of files, which is
# exactly what the old per-sample images/ folder was. The packed .npz files
# below are each a single file, so a plain stat call is safe.
for label, path in [('spectrograms_train.npz', SPEC_TRAIN_PATH),
                    ('spectrograms_val.npz',   SPEC_VAL_PATH),
                    ('spectrograms_test.npz',  SPEC_TEST_PATH),
                    ('image_list.csv',         MANIFEST_CSV),
                    ('freq_axis.npy',          FREQ_AXIS_PATH),
                    ('time_axis.npy',          TIME_AXIS_PATH)]:
    exists = os.path.exists(path)
    size_mb = f'  ({os.path.getsize(path) / 1e6:.1f} MB)' if exists else ''
    print(f"{'OK' if exists else 'MISSING'}  {label:24s}  {path}{size_mb}")

## Cell 3b — Set up a persistent run log

Everything printed from this cell onward (data shapes, class weights, per-epoch
training summaries, the classification report, confusion matrix) is duplicated to
a timestamped text file in `LOG_DIR`, alongside the other saved artifacts
(`best_model.keras`, `confusion_matrix.png`, etc.) -- no more manually copy-pasting
console output after a run.

In [ ]:
import sys
import datetime

class _Tee:
    """Duplicates every write to both the live notebook output and a log file."""
    def __init__(self, *streams):
        self.streams = streams
    def write(self, data):
        for s in self.streams:
            s.write(data)
    def flush(self):
        for s in self.streams:
            s.flush()

RUN_LOG_PATH = os.path.join(LOG_DIR, f"run_log_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}.txt")
_log_file = open(RUN_LOG_PATH, 'w', encoding='utf-8')
sys.stdout = _Tee(sys.__stdout__, _log_file)   # sys.__stdout__ (not sys.stdout) so reruns don't nest tees

print(f"{'='*70}")
print(f"  07b training run -- {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"  EPOCHS={EPOCHS}  BATCH_SIZE={BATCH_SIZE}  LEARNING_RATE={LEARNING_RATE}  DROPOUT_RATE={DROPOUT_RATE}")
print(f"  CLASS_NAMES={CLASS_NAMES}")
print(f"{'='*70}")
print(f"\nLogging everything printed from here on -> {RUN_LOG_PATH}\n")

## Cell 4 — Check TensorFlow + GPU visibility

In [ ]:
import tensorflow as tf
print(f'TensorFlow : {tf.__version__}')
gpus = tf.config.list_physical_devices('GPU')
print(f'GPUs       : {gpus}')
if not gpus:
    print('WARNING: no GPU — go to Runtime > Change runtime type > T4 GPU')

## Cell 5 — Load manifest and inspect class / split distribution

Splits (train/val/test) were already assigned by 07a on the cluster, by EVENT
(stratified by class) — this notebook just reads the `split` column, no re-splitting.

In [ ]:
import pandas as pd
import numpy as np

manifest = pd.read_csv(MANIFEST_CSV)
print(f'Manifest: {len(manifest):,} rows')

label2idx = {name: i for i, name in enumerate(CLASS_NAMES)}
idx2label = {i: name for name, i in label2idx.items()}
print('Label encoding:', label2idx)

print('\nRows per split x class:')
print(manifest.groupby(['split', 'event_type']).size().unstack(fill_value=0).to_string())

freq_axis = np.load(FREQ_AXIS_PATH)
time_axis = np.load(TIME_AXIS_PATH)
print(f'\nFrequency axis: {len(freq_axis)} bins, 0-{freq_axis.max():.1f} Hz')
print(f'Time axis     : {len(time_axis)} bins, 0-{time_axis.max():.1f} s')

## Cell 6 — Load packed spectrogram archives into memory

Each split (train/val/test) is one consolidated `.npz` file built by
`07a_consolidate_for_colab.py` — a single `np.load()` per split instead of
tens of thousands of individual file reads.

Arrays are kept in **float16** (the dtype they were packed in) rather than
upcast to float32 here — upcasting the whole dataset immediately doubles its
RAM footprint before you've even started using it. The model still trains in
float32; the cast happens per-batch inside the tf.data pipeline in Cell 8,
so only one batch at a time is ever float32, not the whole dataset.

In [ ]:
def load_packed_split(path):
    with np.load(path, allow_pickle=False) as d:
        X = d['images']   # float16, as packed by 07a_consolidate_for_colab.py -- NOT upcast here
        y = np.array([label2idx[lbl] for lbl in d['labels']], dtype='int32')
    return X, y

X_train, y_train = load_packed_split(SPEC_TRAIN_PATH)
X_val,   y_val   = load_packed_split(SPEC_VAL_PATH)
X_test,  y_test  = load_packed_split(SPEC_TEST_PATH)

print(f'Train: X={X_train.shape}  y={y_train.shape}  dtype={X_train.dtype}  ({X_train.nbytes / 1e9:.2f} GB)')
print(f'Val  : X={X_val.shape}  y={y_val.shape}  dtype={X_val.dtype}  ({X_val.nbytes / 1e9:.2f} GB)')
print(f'Test : X={X_test.shape}  y={y_test.shape}  dtype={X_test.dtype}  ({X_test.nbytes / 1e9:.2f} GB)')

INPUT_SHAPE = X_train.shape[1:]
print(f'\nCNN input shape: {INPUT_SHAPE}')

## Cell 6b — Sanity-check the packed data (diagnostic)

Training has collapsed to predicting a single constant class regardless of learning
rate (1e-3, 3e-4) or batch size (32, 128), and train accuracy hasn't moved above the
~33% random baseline for 3 classes after several epochs. That combination points at
the data, not the hyperparameters -- either the images carry no learnable signal, or
there's a label/image alignment bug somewhere upstream (07a extraction or the
consolidate/pack step). This cell checks three things directly on the raw arrays,
before Cell 7 normalizes them in place:

1. Are any images all-zero placeholders? (`07a_consolidate_for_colab.py` zero-fills
   a sample if its `.npz` failed to load -- if that happened for a large fraction of
   samples, the model is literally training on blank inputs for those samples.)
2. Do per-class raw pixel statistics (mean/std/min/max) actually differ at all?
   (A coarse check -- real signal can still be spatial even if global stats are
   similar, but near-identical stats across all 3 classes is a yellow flag.)
3. Does the labels array line up with the manifest's row order for each split?
   (If not, the model is being trained/evaluated against the wrong labels entirely.)

**Memory note**: computed in small fixed-size chunks (1024 samples at a time), not
via `X[mask].astype('float32')` on the whole class subset at once -- earthquake alone
is 83% of train, so a naive full-subset float32 copy would allocate ~5 GB in one shot,
which is almost certainly why this cell crashed on the first pass.

In [ ]:
import gc

def chunked_stats(X, idx, chunk=1024):
    """Mean/std/min/max over X[idx], processed in small chunks so peak memory
    stays bounded regardless of how large the class is (no full-subset copy)."""
    total = 0.0
    total_sq = 0.0
    vmin, vmax = np.inf, -np.inf
    count = 0
    for start in range(0, len(idx), chunk):
        block = X[idx[start:start + chunk]].astype('float32')
        total += float(block.sum())
        total_sq += float((block ** 2).sum())
        vmin = min(vmin, float(block.min()))
        vmax = max(vmax, float(block.max()))
        count += block.size
        del block
    mean = total / count
    var = max(total_sq / count - mean ** 2, 0.0)
    return mean, var ** 0.5, vmin, vmax

def check_split(name, X, y, manifest_split):
    n = len(y)
    # .any(axis=1) is a reduction -- no full-size boolean copy of X, unlike `X == 0`.
    has_nonzero = X.reshape(n, -1).any(axis=1)
    n_zero = int(n - has_nonzero.sum())
    print(f'--- {name} ---')
    print(f'  Zero-filled images (failed loads in consolidate step): {n_zero}/{n} ({100*n_zero/n:.2f}%)')
    del has_nonzero

    for cls_idx, cls_name in idx2label.items():
        idx = np.where(y == cls_idx)[0]
        if len(idx) == 0:
            continue
        mean, std, vmin, vmax = chunked_stats(X, idx)
        print(f'  {cls_name:12s} n={len(idx):6d}  mean={mean:8.3f}  std={std:8.3f}  '
              f'min={vmin:8.3f}  max={vmax:8.3f}')

    # Order check: manifest rows for this split (same filter + reset_index used by
    # 07a_consolidate_for_colab.py when it packed the arrays) vs the loaded labels.
    manifest_labels = manifest_split['event_type'].values
    loaded_labels = np.array([idx2label[i] for i in y])
    match = np.mean(manifest_labels[:n] == loaded_labels[:len(manifest_labels)])
    print(f'  Label order match vs manifest: {match*100:.1f}%  (should be ~100%)')
    gc.collect()

for split_name, X, y in [('train', X_train, y_train), ('val', X_val, y_val), ('test', X_test, y_test)]:
    manifest_split = manifest[manifest['split'] == split_name].reset_index(drop=True)
    check_split(split_name, X, y, manifest_split)

## Cell 7 — Per-channel normalization (fit on TRAIN only)

Z-score each of the 3 channels (Z, N, E spectrograms) independently, using statistics
computed on the training set only — never fit normalization on val/test, that would leak
information. Stats are saved to Drive so the exact same normalization can be reused at
inference time on new spectrograms.

Memory note: `X_train` and `X_val` are normalized **in place** (`-=` / `/=`), staying
float16, instead of creating a second full-size array — their raw values aren't needed
again after this cell. `X_test` is the exception: Cell 14 (Grad-CAM) displays the raw
values AND feeds the normalized version straight to the model outside the tf.data
pipeline, so it keeps both — cheap, since test is the smallest split.

In [ ]:
# dtype='float32' forces the mean/std reduction to accumulate in float32 even
# though X_train is float16 -- summing tens of thousands of float16 values
# directly would lose meaningful precision. The result itself is tiny
# (shape (1, 1, 1, 3)), so this costs nothing.
channel_mean = X_train.mean(axis=(0, 1, 2), keepdims=True, dtype='float32')
channel_std  = X_train.std(axis=(0, 1, 2), keepdims=True, dtype='float32') + 1e-8

np.savez(os.path.join(LOG_DIR, 'normalization_stats.npz'),
         mean=channel_mean, std=channel_std)
print('Per-channel mean:', channel_mean.ravel())
print('Per-channel std :', channel_std.ravel())

# Normalize X_train / X_val IN PLACE, staying float16 -- avoids ever holding
# a second full-size copy of the two largest arrays in memory at once.
mean16 = channel_mean.astype('float16')
std16  = channel_std.astype('float16')

X_train -= mean16
X_train /= std16
X_val   -= mean16
X_val   /= std16

X_train_n = X_train   # normalized in place -- same array, just named for clarity
X_val_n   = X_val

# X_test: keep the raw version (Cell 14 displays it) AND build a normalized
# float32 copy (Grad-CAM feeds it directly to the model, bypassing the
# tf.data cast in Cell 8) -- test is the smallest split, so this is cheap.
X_test_n = (X_test.astype('float32') - channel_mean) / channel_std

print(f'\nSaved -> normalization_stats.npz')
print(f'X_train_n: dtype={X_train_n.dtype}  ({X_train_n.nbytes / 1e9:.2f} GB)')
print(f'X_val_n  : dtype={X_val_n.dtype}  ({X_val_n.nbytes / 1e9:.2f} GB)')
print(f'X_test_n : dtype={X_test_n.dtype}  ({X_test_n.nbytes / 1e9:.2f} GB)')

## Cell 8 — Data augmentation (SpecAugment-style) + tf.data pipeline

Replaces SMOTE (which doesn't map onto images): random time masking, random frequency
masking, and small amplitude jitter, applied only to the training set.

**Two more memory issues, both caused by `tf.data.Dataset.from_generator` (the previous
fix for Cell 8's crash), fixed here:**

1. **The "Your input ran out of data" warning.** `from_generator` gives the dataset an
   *unknown* cardinality -- Keras doesn't know how many batches an epoch has ahead of
   time, so on epoch 1 it iterates until the generator is exhausted, warns, then locks
   in the true step count for later epochs. Mostly cosmetic on its own.
2. **RAM climbing epoch over epoch.** A known TensorFlow issue specific to
   `from_generator`: the live Python generator's state isn't fully released between
   epochs, so it slowly accumulates -- this is what was heading toward another crash
   mid-training.

**Fix**: build the dataset from a plain integer index range (`tf.data.Dataset.range(n)`,
which has a *known*, finite size -- no more warning) and fetch each sample lazily via
`tf.numpy_function`, which carries no per-epoch generator state. Since we're now only
shuffling small integers rather than full images, the shuffle buffer can safely cover
the entire training set again at negligible memory cost, so shuffling quality is back
to a true full-dataset shuffle.

In [ ]:
def spec_augment(image, label, n_time_masks=1, n_freq_masks=1,
                 max_time_mask_frac=0.15, max_freq_mask_frac=0.15, noise_std=0.05):
    img = tf.identity(image)
    n_freq  = tf.shape(img)[0]
    n_time  = tf.shape(img)[1]

    for _ in range(n_time_masks):
        mask_w = tf.random.uniform([], 0, tf.cast(tf.cast(n_time, tf.float32) * max_time_mask_frac, tf.int32) + 1, dtype=tf.int32)
        mask_w = tf.maximum(mask_w, 1)
        t0 = tf.random.uniform([], 0, tf.maximum(n_time - mask_w, 1), dtype=tf.int32)
        time_idx = tf.range(n_time)
        time_mask = tf.logical_and(time_idx >= t0, time_idx < t0 + mask_w)
        time_mask = tf.cast(tf.logical_not(time_mask), img.dtype)[tf.newaxis, :, tf.newaxis]
        img = img * time_mask

    for _ in range(n_freq_masks):
        mask_h = tf.random.uniform([], 0, tf.cast(tf.cast(n_freq, tf.float32) * max_freq_mask_frac, tf.int32) + 1, dtype=tf.int32)
        mask_h = tf.maximum(mask_h, 1)
        f0 = tf.random.uniform([], 0, tf.maximum(n_freq - mask_h, 1), dtype=tf.int32)
        freq_idx = tf.range(n_freq)
        freq_mask = tf.logical_and(freq_idx >= f0, freq_idx < f0 + mask_h)
        freq_mask = tf.cast(tf.logical_not(freq_mask), img.dtype)[:, tf.newaxis, tf.newaxis]
        img = img * freq_mask

    img = img + tf.random.normal(tf.shape(img), mean=0.0, stddev=noise_std, dtype=img.dtype)
    return img, label

def to_float32(image, label):
    return tf.cast(image, tf.float32), label

AUTOTUNE = tf.data.AUTOTUNE

# NOTE: from_generator (previous version of this cell) has two downsides:
# 1) unknown dataset cardinality -> Keras' "ran out of data" warning on epoch 1
# 2) a known TensorFlow issue where the live Python generator's state isn't
#    fully released between epochs, causing RAM to climb during training.
# Fix: index the arrays via a plain int range dataset (known size, trivial
# memory) + tf.numpy_function to fetch each sample lazily -- no generator,
# no per-epoch state, no duplicated arrays.

def make_lookup_ds(X, y, img_dtype, shuffle_buffer=None, seed=None):
    n = len(y)
    ds = tf.data.Dataset.range(n)
    if shuffle_buffer:
        ds = ds.shuffle(buffer_size=shuffle_buffer, seed=seed)

    def _fetch(i):
        return X[i], y[i]

    def fetch_fn(i):
        img, lbl = tf.numpy_function(_fetch, [i], [img_dtype, tf.int32])
        img.set_shape(INPUT_SHAPE)
        lbl.set_shape([])
        return img, lbl

    return ds.map(fetch_fn, num_parallel_calls=AUTOTUNE)

# Shuffling only integer indices now (not images), so a full-size buffer is
# effectively free -- back to a true per-epoch shuffle of the whole training set.
SHUFFLE_BUFFER = len(y_train)

train_ds = (make_lookup_ds(X_train_n, y_train, tf.float16, shuffle_buffer=SHUFFLE_BUFFER, seed=42)
            .map(spec_augment, num_parallel_calls=AUTOTUNE)
            .map(to_float32, num_parallel_calls=AUTOTUNE)
            .batch(BATCH_SIZE)
            .prefetch(AUTOTUNE))

val_ds = (make_lookup_ds(X_val_n, y_val, tf.float16)
          .map(to_float32, num_parallel_calls=AUTOTUNE)
          .batch(BATCH_SIZE)
          .prefetch(AUTOTUNE))

test_ds = (make_lookup_ds(X_test_n, y_test, tf.float32)
           .map(to_float32, num_parallel_calls=AUTOTUNE)
           .batch(BATCH_SIZE)
           .prefetch(AUTOTUNE))

print('tf.data pipelines ready.')

## Cell 9 — Build the CNN

Shallow on purpose: this dataset is small (hundreds-to-low-thousands of samples), so a
deep network would overfit fast. GlobalAveragePooling2D instead of Flatten+Dense keeps
the parameter count low. Strided convolutions handle downsampling (no separate pooling
layers), same style as the depthwise conv blocks in `deepdenoiser/model.py`.

**Two backbone options** (set `MODEL_BACKBONE` in Cell 3):

- `'custom'` — the shallow from-scratch CNN below (`build_cnn`).
- `'resnet50'` — an ImageNet-pretrained ResNet50 backbone, fine-tuned on our
  spectrograms (`build_resnet50_transfer`). Loosely follows the transfer-learning
  strategy from Dong et al. (2023) *"Microseismic event waveform classification
  using CNN-based transfer learning models"*: take a CNN pretrained on natural
  images, freeze the early/mid layers (generic edge/texture filters), and only
  fine-tune the last conv block plus a brand-new classification head sized for
  our 3 classes. Differences from their setup: they classified 4 classes from
  RGB *plots* of 6-channel waveforms resized to 224x224/227x227; we classify
  3 classes directly from our (116, 117, 3) dB-spectrogram images at native
  resolution — no resizing needed since `tf.keras.applications` models accept
  arbitrary input sizes (min 32x32) when `include_top=False`. They also picked
  GoogLeNet as their best-performing backbone; GoogLeNet/AlexNet aren't
  available in `tf.keras.applications`, so ResNet50 (which they also tested,
  and which is fully supported here) is the practical choice.

In [ ]:
from tensorflow.keras import layers, models

def build_cnn(input_shape, n_classes, dropout_rate=0.5):
    inputs = layers.Input(shape=input_shape, name='spectrogram')

    x = inputs
    for i, filters in enumerate([32, 64, 128]):
        x = layers.Conv2D(filters, 3, padding='same', use_bias=False, name=f'conv{i+1}_a')(x)
        x = layers.BatchNormalization(name=f'bn{i+1}_a')(x)
        x = layers.Activation('relu', name=f'relu{i+1}_a')(x)
        x = layers.Conv2D(filters, 3, strides=2, padding='same', use_bias=False, name=f'conv{i+1}_down')(x)
        x = layers.BatchNormalization(name=f'bn{i+1}_down')(x)
        x = layers.Activation('relu', name=f'relu{i+1}_down')(x)

    x = layers.GlobalAveragePooling2D(name='gap')(x)
    x = layers.Dropout(dropout_rate, name='dropout')(x)
    outputs = layers.Dense(n_classes, activation='softmax', name='predictions')(x)

    return models.Model(inputs, outputs, name='spectrogram_cnn')


def build_resnet50_transfer(input_shape, n_classes, dropout_rate=0.5, unfreeze_last_n=15):
    """ImageNet-pretrained ResNet50 backbone + new head, fine-tuned on our
    spectrograms. Strategy follows Dong et al. (2023): freeze most of the
    backbone (generic low/mid-level filters transfer reasonably well even from
    natural images), fine-tune only the last conv block, and replace the
    classification head entirely (their '1000-way ImageNet head' -> our
    3-way spectrogram head, same idea as their 'replace the last three
    layers' approach for AlexNet/GoogLeNet/ResNet50).

    `pooling='avg'` bakes a GlobalAveragePooling2D into the backbone output,
    so we go straight from base.output (shape (batch, 2048)) to Dropout+Dense
    -- no extra pooling layer needed.
    """
    base = tf.keras.applications.ResNet50(
        include_top=False,
        weights='imagenet',
        input_shape=input_shape,
        pooling='avg',
    )
    base.trainable = True
    for layer in base.layers[:-unfreeze_last_n]:
        layer.trainable = False
    n_trainable = sum(1 for l in base.layers if l.trainable)
    print(f'ResNet50 backbone: {n_trainable}/{len(base.layers)} layers trainable '
          f'(last {unfreeze_last_n} unfrozen)')

    x = layers.Dropout(dropout_rate, name='dropout')(base.output)
    outputs = layers.Dense(n_classes, activation='softmax', name='predictions')(x)

    return models.Model(base.input, outputs, name='spectrogram_resnet50_transfer')


if MODEL_BACKBONE == 'resnet50':
    model = build_resnet50_transfer(INPUT_SHAPE, len(CLASS_NAMES), dropout_rate=DROPOUT_RATE,
                                     unfreeze_last_n=RESNET_UNFREEZE_LAST_N)
    active_lr = FINE_TUNE_LEARNING_RATE
elif MODEL_BACKBONE == 'custom':
    model = build_cnn(INPUT_SHAPE, len(CLASS_NAMES), dropout_rate=DROPOUT_RATE)
    active_lr = LEARNING_RATE
else:
    raise ValueError(f"Unknown MODEL_BACKBONE={MODEL_BACKBONE!r}, expected 'custom' or 'resnet50'")

print(f'MODEL_BACKBONE={MODEL_BACKBONE!r}  learning_rate={active_lr}')
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=active_lr),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)
model.summary()


## Cell 10 — Class weights + callbacks

`class_weight='balanced'` is the CNN-training analog of `RF_CLASS_WEIGHT='balanced'` in
`06a_train_RF_classifier.py` — SMOTE doesn't apply to images, so this (plus the
augmentation in Cell 8) is how class imbalance gets handled here.

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

class_weight_values = compute_class_weight('balanced', classes=np.arange(len(CLASS_NAMES)), y=y_train)
class_weight_dict = {i: w for i, w in enumerate(class_weight_values)}
print('Class weights:', {idx2label[i]: round(w, 3) for i, w in class_weight_dict.items()})

checkpoint_path = os.path.join(LOG_DIR, 'best_model.keras')
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=12, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint(checkpoint_path, monitor='val_loss', save_best_only=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6),
]
print(f'Checkpoints -> {checkpoint_path}')

## Cell 11 — Train

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    class_weight=class_weight_dict,
    callbacks=callbacks,
    verbose=2,   # one clean summary line per epoch -- verbose=1's live-updating
                 # progress bar writes many overlapping partial-progress lines
                 # when captured to a plain text log instead of a real terminal.
)

# Persist the full per-epoch history so plots/reports can be regenerated later
# without retraining.
history_path = os.path.join(LOG_DIR, 'training_history.csv')
pd.DataFrame(history.history).to_csv(history_path, index_label='epoch')
print(f'\n[SAVED] {history_path}')

best_epoch = int(np.argmin(history.history['val_loss']))
print(f"Best epoch (lowest val_loss): {best_epoch + 1}  "
      f"(val_loss={history.history['val_loss'][best_epoch]:.4f}, "
      f"val_accuracy={history.history['val_accuracy'][best_epoch]:.4f})")
print("(EarlyStopping restore_best_weights=True -> this is the model now in memory.)")

## Cell 12 — Training curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(history.history['loss'], label='train')
axes[0].plot(history.history['val_loss'], label='val')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss'); axes[0].set_title('Loss')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['accuracy'], label='train')
axes[1].plot(history.history['val_accuracy'], label='val')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy'); axes[1].set_title('Accuracy')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
curves_path = os.path.join(LOG_DIR, 'training_curves.png')
plt.savefig(curves_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'[SAVED] {curves_path}')

# Standalone loss-only plot -- easier to drop into a report on its own.
fig_loss, ax_loss = plt.subplots(figsize=(7, 5))
ax_loss.plot(history.history['loss'], label='train')
ax_loss.plot(history.history['val_loss'], label='val')
ax_loss.set_xlabel('Epoch'); ax_loss.set_ylabel('Loss')
ax_loss.set_title('Loss evolution over training')
ax_loss.legend(); ax_loss.grid(True, alpha=0.3)
plt.tight_layout()
loss_only_path = os.path.join(LOG_DIR, 'loss_curve.png')
plt.savefig(loss_only_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'[SAVED] {loss_only_path}')

## Cell 13 — Evaluate on test set

Same evaluation suite as `06a_train_RF_classifier.py`: classification report, confusion
matrix, one-vs-rest ROC curves — so CNN and RF/HGB results are directly comparable.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc
from sklearn.preprocessing import label_binarize

y_proba = model.predict(test_ds)
y_pred  = np.argmax(y_proba, axis=1)

print(classification_report(y_test, y_pred, target_names=CLASS_NAMES))

cm = confusion_matrix(y_test, y_pred, labels=np.arange(len(CLASS_NAMES)))
print('Confusion matrix (rows=true, cols=predicted):')
print(pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES).to_string())

fig_cm, ax_cm = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES).plot(ax=ax_cm, cmap='Blues', colorbar=False)
ax_cm.set_title('Confusion matrix — test set (CNN)')
plt.tight_layout()
cm_path = os.path.join(LOG_DIR, 'confusion_matrix.png')
plt.savefig(cm_path, dpi=150)
plt.show()
print(f'[SAVED] {cm_path}')

y_test_bin = label_binarize(y_test, classes=np.arange(len(CLASS_NAMES)))
fig_roc, ax_roc = plt.subplots(figsize=(7, 5))
for i, name in enumerate(CLASS_NAMES):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_proba[:, i])
    ax_roc.plot(fpr, tpr, lw=2, label=f'{name}  (AUC={auc(fpr, tpr):.3f})')
ax_roc.plot([0, 1], [0, 1], 'k--', lw=1)
ax_roc.set_xlabel('False Positive Rate'); ax_roc.set_ylabel('True Positive Rate')
ax_roc.set_title('ROC curves — one-vs-rest (test set, CNN)')
ax_roc.legend(loc='lower right')
plt.tight_layout()
roc_path = os.path.join(LOG_DIR, 'roc_curves.png')
plt.savefig(roc_path, dpi=150)
plt.show()
print(f'[SAVED] {roc_path}')

## Cell 14 — Grad-CAM on example test images

Shows which time-frequency region the CNN is keying on for a few correctly-classified
examples per class — a domain-expert sanity check that the network is reading actual
event signatures (e.g. a rockslide's low-frequency coda) rather than an artifact.

In [ ]:
last_conv_name = None
for layer in model.layers:
    if isinstance(layer, layers.Conv2D):
        last_conv_name = layer.name
print(f'Grad-CAM target layer: {last_conv_name}')

grad_model = tf.keras.models.Model(model.inputs, [model.get_layer(last_conv_name).output, model.output])

def grad_cam(img_batch, class_idx):
    with tf.GradientTape() as tape:
        conv_out, preds = grad_model(img_batch)
        loss = preds[:, class_idx]
    grads = tape.gradient(loss, conv_out)
    weights = tf.reduce_mean(grads, axis=(1, 2), keepdims=True)
    cam = tf.reduce_sum(weights * conv_out, axis=-1)
    cam = tf.nn.relu(cam)
    cam = cam / (tf.reduce_max(cam, axis=(1, 2), keepdims=True) + 1e-8)
    return cam.numpy()

fig, axes = plt.subplots(len(CLASS_NAMES), 2, figsize=(9, 3.2 * len(CLASS_NAMES)))
for row, name in enumerate(CLASS_NAMES):
    idx_candidates = np.where((y_test == label2idx[name]) & (y_pred == label2idx[name]))[0]
    if len(idx_candidates) == 0:
        for col in range(2):
            axes[row, col].text(0.5, 0.5, f'No correct\n{name} example', ha='center', va='center')
            axes[row, col].axis('off')
        continue
    sample_idx = idx_candidates[0]
    img_batch = X_test_n[sample_idx:sample_idx + 1]
    cam = grad_cam(img_batch, label2idx[name])[0]

    axes[row, 0].imshow(X_test[sample_idx][:, :, 0], aspect='auto', origin='lower', cmap='viridis')
    axes[row, 0].set_title(f'{name} — Z channel (raw dB)')
    axes[row, 1].imshow(X_test[sample_idx][:, :, 0], aspect='auto', origin='lower', cmap='gray')
    axes[row, 1].imshow(cam, aspect='auto', origin='lower', cmap='jet', alpha=0.45,
                        extent=axes[row, 1].get_xlim() + axes[row, 1].get_ylim())
    axes[row, 1].set_title(f'{name} — Grad-CAM overlay')

plt.tight_layout()
gradcam_path = os.path.join(LOG_DIR, 'grad_cam_examples.png')
plt.savefig(gradcam_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'[SAVED] {gradcam_path}')

## Cell 15 — Save final model

In [ ]:
final_model_path = os.path.join(LOG_DIR, 'spectrogram_cnn_final.keras')
model.save(final_model_path)
print(f'[SAVED] {final_model_path}')
print(f'\nReload with:')
print(f"  model = tf.keras.models.load_model('{final_model_path}')")
print(f'\nDon\'t forget: apply the SAME normalization at inference time —')
print(f"  stats = np.load('{os.path.join(LOG_DIR, 'normalization_stats.npz')}')")
print(f"  X_new_norm = (X_new - stats['mean']) / stats['std']")

## Cell 16 — Close the run log

In [ ]:
print(f"\n{'='*70}")
print(f"  Run log saved -> {RUN_LOG_PATH}")
print(f"{'='*70}")

sys.stdout = sys.__stdout__
_log_file.close()